# Synthetic Q&A Generation with Reasoning (CoT Self-Instruct)

## Method Overview

**CoT Self-Instruct** combines self-instruct (generating new questions from seeds) with Chain-of-Thought reasoning.

### The Problem You Had:
Using Qwen 14B to generate questions → It only created questions it could already answer

### The Solution:
Use a **more capable model** (Claude Sonnet 4, GPT-4o, or Qwen 32B) to generate questions that will challenge Qwen 14B

### Our Approach:
1. **Sample seed examples** from your cleaned dataset
2. **Generate similar questions** using a stronger model
3. **Solve with reasoning** using the same strong model
4. **Quality filter** the outputs
5. **Format for fine-tuning** Qwen 14B

### Key Insight:
The stronger model can create questions and solutions that are *slightly above* Qwen 14B's current capability, 
which is exactly what you want for effective fine-tuning!

## Setup

In [ ]:
import json
import random
import sys
from pathlib import Path
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, field
from openai import OpenAI
from tqdm.notebook import tqdm
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import time
import re

sys.path.insert(0, '..')
from utils.api_client import create_openrouter_client
from utils.data_io import load_dataset, save_dataset

# Initialize OpenRouter client
client = create_openrouter_client()

# Token tracking
total_tokens = 0
token_lock = threading.Lock()

## Configuration

In [ ]:
@dataclass
class SyntheticConfig:
    """Configuration for synthetic data generation"""
    # Input/Output
    input_file: str = "../data/seed_dataset.json"
    output_file: str = "../data/synthetic_qra_dataset.jsonl"
    finetuning_file: str = "../data/synthetic_finetuning_format.jsonl"
    
    # Model selection
    # Use a MORE CAPABLE model than what you're fine-tuning
    # Options: "anthropic/claude-sonnet-4", "openai/gpt-4o", "qwen/qwen-2.5-32b-instruct"
    generator_model: str = "anthropic/claude-sonnet-4"  # Stronger model for generation
    
    # Generation parameters
    num_synthetic: int = 50  # Total synthetic examples to generate
    num_seed_examples: int = 3  # How many examples to show as context
    questions_per_seed: int = 2  # Generate 2 questions per seed set
    
    # Quality thresholds
    min_question_length: int = 30
    min_reasoning_length: int = 50
    min_answer_length: int = 1
    
    # Parallel processing
    max_workers: int = 5
    temperature: float = 0.8  # Higher for diversity
    max_tokens: int = 25000

config = SyntheticConfig()

print(f"Configuration:")
print(f"  Input: {config.input_file}")
print(f"  Generator model: {config.generator_model}")
print(f"  Target synthetic examples: {config.num_synthetic}")
print(f"  Seed examples per prompt: {config.num_seed_examples}")
print(f"  Questions per seed set: {config.questions_per_seed}")

## Load Seed Dataset

In [ ]:
# Load seed examples (uses shared load_dataset from utils.data_io)
seed_dataset = load_dataset(config.input_file)
print(f"Loaded {len(seed_dataset)} seed examples")

# Show sample
if seed_dataset:
    print(f"\nSample seed:")
    sample = seed_dataset[0]
    print(f"  Q: {sample['question'][:100]}...")
    print(f"  R: {sample['reasoning'][:100]}...")
    print(f"  A: {sample['answer'][:50]}...")

## Prompts for Generation

The key is to show the model examples and ask it to generate SIMILAR but NEW questions.

In [4]:
GENERATION_PROMPT = """You are an expert in reliability engineering, creating practice problems for students.

I will show you {num_examples} example problems. Your task is to:
1. Understand the STYLE, DIFFICULTY, and TOPICS of these examples
2. Create {num_to_generate} NEW, ORIGINAL problems that are SIMILAR in style and difficulty
3. Make questions that are EDUCATIONAL and CHALLENGING
4. Provide COMPLETE solutions with step-by-step reasoning

IMPORTANT:
- Create DIFFERENT questions (don't just change numbers in the examples)
- Use similar concepts but NEW scenarios
- Include all necessary data in the question (self-contained)
- Show clear reasoning steps
- Provide specific numerical answers when appropriate

# EXAMPLE PROBLEMS:

{examples}

# YOUR TASK:

Generate {num_to_generate} NEW problems inspired by these examples. For each problem, provide:

QUESTION: [The complete problem statement with all necessary data]

REASONING: [Step-by-step solution showing your work]

ANSWER: [Final answer(s) - be specific]

---

Now generate the {num_to_generate} new problems:"""


def format_examples(examples: List[Dict]) -> str:
    """Format seed examples for the prompt"""
    formatted = []
    for i, ex in enumerate(examples, 1):
        formatted.append(f"""## Example {i}:

QUESTION: {ex['question']}

REASONING: {ex['reasoning']}

ANSWER: {ex['answer']}
""")
    return "\n".join(formatted)


def create_generation_prompt(seed_examples: List[Dict], config: SyntheticConfig) -> str:
    """Create the full generation prompt"""
    examples_text = format_examples(seed_examples)
    
    return GENERATION_PROMPT.format(
        num_examples=len(seed_examples),
        num_to_generate=config.questions_per_seed,
        examples=examples_text
    )


# Test the prompt
test_seeds = seed_dataset[:3]
test_prompt = create_generation_prompt(test_seeds, config)

print("Sample generation prompt (first 800 chars):")
print("="*80)
print(test_prompt[:800])
print("...")
print("="*80)

Sample generation prompt (first 800 chars):
You are an expert in reliability engineering, creating practice problems for students.

I will show you 3 example problems. Your task is to:
1. Understand the STYLE, DIFFICULTY, and TOPICS of these examples
2. Create 2 NEW, ORIGINAL problems that are SIMILAR in style and difficulty
3. Make questions that are EDUCATIONAL and CHALLENGING
4. Provide COMPLETE solutions with step-by-step reasoning

IMPORTANT:
- Create DIFFERENT questions (don't just change numbers in the examples)
- Use similar concepts but NEW scenarios
- Include all necessary data in the question (self-contained)
- Show clear reasoning steps
- Provide specific numerical answers when appropriate

# EXAMPLE PROBLEMS:

## Example 1:

QUESTION: Suppose there is one spare unit available for replacement, the operating unit is repla
...


## Generation Functions

In [5]:
def call_generator_model(prompt: str, config: SyntheticConfig) -> str:
    """Call the generator model"""
    global total_tokens
    
    try:
        response = client.chat.completions.create(
            model=config.generator_model,
            max_tokens=config.max_tokens,
            temperature=config.temperature,
            messages=[{"role": "user", "content": prompt}]
        )
        
        # Track tokens
        with token_lock:
            total_tokens += response.usage.prompt_tokens + response.usage.completion_tokens
        
        return response.choices[0].message.content
        
    except Exception as e:
        print(f"Error calling model: {e}")
        return ""


def parse_generated_output(output: str) -> List[Dict]:
    """
    Parse the model's output to extract Q/R/A triplets.
    
    Returns list of {question, reasoning, answer} dicts
    """
    results = []
    
    # Split by "QUESTION:" markers (case insensitive)
    sections = [s.strip() for s in output.split('QUESTION:') if s.strip()]
    
    for section in sections:
        # Extract reasoning and answer
        reasoning_match = re.search(
            r'REASONING:\s*(.*?)(?=ANSWER:|$)', 
            section, 
            re.IGNORECASE | re.DOTALL
        )
        
        answer_match = re.search(
            r'ANSWER:\s*(.*?)(?=QUESTION:|---+|$)', 
            section, 
            re.IGNORECASE | re.DOTALL
        )
        
        # Extract question (before REASONING: or ANSWER:)
        question_text = re.split(
            r'(REASONING|ANSWER):', 
            section, 
            flags=re.IGNORECASE, 
            maxsplit=1
        )[0].strip()
        
        reasoning_text = reasoning_match.group(1).strip() if reasoning_match else ""
        answer_text = answer_match.group(1).strip() if answer_match else ""
        
        # Only add if we have all three components
        if question_text and reasoning_text and answer_text:
            results.append({
                'question': question_text,
                'reasoning': reasoning_text,
                'answer': answer_text
            })
    
    return results


def validate_generated(item: Dict, config: SyntheticConfig) -> Tuple[bool, str]:
    """Validate a generated Q/R/A triplet"""
    # Check lengths
    if len(item['question']) < config.min_question_length:
        return False, "Question too short"
    
    if len(item['reasoning']) < config.min_reasoning_length:
        return False, "Reasoning too short"
    
    if len(item['answer']) < config.min_answer_length:
        return False, "Answer too short"
    
    # Check for placeholder text
    invalid_patterns = ['[insert', 'tbd', 'to be determined', 'xxx', '???']
    text = (item['question'] + item['reasoning'] + item['answer']).lower()
    
    for pattern in invalid_patterns:
        if pattern in text:
            return False, f"Contains placeholder: {pattern}"
    
    return True, ""


print("✓ Generation functions defined")

✓ Generation functions defined


## Test Generation

In [6]:
import re

# Test with a small sample
print("Testing generation with 3 seed examples...\n")

test_seeds = random.sample(seed_dataset, 3)
test_prompt = create_generation_prompt(test_seeds, config)

print("Calling generator model...")
test_output = call_generator_model(test_prompt, config)

print(f"\nReceived {len(test_output)} characters\n")
print("Raw output (first 500 chars):")
print("="*80)
print(test_output[:500])
print("...")
print("="*80)

# Parse the output
test_parsed = parse_generated_output(test_output)

print(f"\nParsed {len(test_parsed)} Q/R/A triplets")

# Show parsed results
for i, item in enumerate(test_parsed, 1):
    print(f"\n--- Generated Question {i} ---")
    print(f"Q: {item['question'][:150]}...")
    print(f"R: {item['reasoning'][:150]}...")
    print(f"A: {item['answer'][:100]}...")
    
    is_valid, reason = validate_generated(item, config)
    print(f"Valid: {is_valid} {f'({reason})' if not is_valid else ''}")

Testing generation with 3 seed examples...

Calling generator model...

Received 3180 characters

Raw output (first 500 chars):
# NEW PROBLEM 1:

**QUESTION:** A manufacturer of industrial pumps offers a warranty policy for their $850 pump units. The warranty provides full replacement for the first 12 months, followed by a pro-rated refund that decreases linearly to zero over the next 48 months. The pump lifetime follows a Weibull distribution with shape parameter β = 2.3 and characteristic life η = 180 months. Calculate the expected warranty cost per pump unit for the manufacturer.

**REASONING:** 
Given data:
- Unit co
...

Parsed 2 Q/R/A triplets

--- Generated Question 1 ---
Q: ** A manufacturer of industrial pumps offers a warranty policy for their $850 pump units. The warranty provides full replacement for the first 12 mont...
R: ** 
Given data:
- Unit cost: $c_p = 850$
- Free replacement period: $t_1 = 12$ months  
- Total warranty period: $t_0 = 12 + 48 = 60$ months
- Weibull..

## Full Generation Pipeline

In [7]:
def generate_batch(seed_examples: List[Dict], config: SyntheticConfig) -> List[Dict]:
    """
    Generate a batch of synthetic Q/R/A pairs from seed examples.
    
    Returns list of validated synthetic examples.
    """
    # Create prompt
    prompt = create_generation_prompt(seed_examples, config)
    
    # Generate
    output = call_generator_model(prompt, config)
    
    if not output:
        return []
    
    # Parse
    parsed = parse_generated_output(output)
    
    # Validate and filter
    valid_examples = []
    for item in parsed:
        is_valid, _ = validate_generated(item, config)
        if is_valid:
            # Add metadata
            item['source'] = 'synthetic'
            item['seed_sources'] = [ex['source'] for ex in seed_examples]
            valid_examples.append(item)
    
    return valid_examples


def generate_synthetic_dataset(seed_dataset: List[Dict], config: SyntheticConfig) -> List[Dict]:
    """
    Generate full synthetic dataset using parallel processing.
    """
    synthetic_data = []
    rejected_count = 0
    
    # Calculate how many batches we need
    # Each batch generates ~questions_per_seed questions
    num_batches = (config.num_synthetic // config.questions_per_seed) + 10  # +10 buffer for rejections
    
    print(f"\n{'='*80}")
    print(f"GENERATING SYNTHETIC DATASET")
    print(f"{'='*80}")
    print(f"Target: {config.num_synthetic} examples")
    print(f"Batches: {num_batches}")
    print(f"Questions per batch: ~{config.questions_per_seed}")
    print(f"Parallel workers: {config.max_workers}")
    print()
    
    start_time = time.time()
    
    with ThreadPoolExecutor(max_workers=config.max_workers) as executor:
        # Create all batch tasks
        futures = []
        for _ in range(num_batches):
            # Sample random seed examples
            seeds = random.sample(seed_dataset, config.num_seed_examples)
            future = executor.submit(generate_batch, seeds, config)
            futures.append(future)
        
        # Collect results with progress bar
        for future in tqdm(
            as_completed(futures), 
            total=len(futures), 
            desc="Generating batches"
        ):
            try:
                batch_results = future.result()
                synthetic_data.extend(batch_results)
                
                # Stop if we've reached target
                if len(synthetic_data) >= config.num_synthetic:
                    break
                    
            except Exception as e:
                print(f"Batch error: {e}")
                rejected_count += 1
    
    # Trim to exact target
    synthetic_data = synthetic_data[:config.num_synthetic]
    
    elapsed = time.time() - start_time
    
    print(f"\n{'='*80}")
    print(f"GENERATION COMPLETE")
    print(f"{'='*80}")
    print(f"Generated: {len(synthetic_data)} examples")
    print(f"Rejected: {rejected_count} batches")
    print(f"Time: {elapsed/60:.1f} minutes")
    print(f"Tokens used: {total_tokens:,}")
    
    return synthetic_data


# RUN GENERATION
synthetic_dataset = generate_synthetic_dataset(seed_dataset, config)


GENERATING SYNTHETIC DATASET
Target: 50 examples
Batches: 35
Questions per batch: ~2
Parallel workers: 5



Generating batches:   0%|          | 0/35 [00:00<?, ?it/s]


GENERATION COMPLETE
Generated: 50 examples
Rejected: 0 batches
Time: 2.7 minutes
Tokens used: 104,648


## Save Results

In [ ]:
def convert_to_finetuning_format(data: List[Dict]) -> List[Dict]:
    """Convert to fine-tuning format with messages"""
    finetuning_data = []
    
    for item in data:
        # Combine reasoning and answer for assistant response
        assistant_content = f"""Let me solve this step by step.

{item['reasoning']}

Therefore, the answer is: {item['answer']}"""
        
        finetuning_data.append({
            "messages": [
                {
                    "role": "user",
                    "content": item['question']
                },
                {
                    "role": "assistant",
                    "content": assistant_content
                }
            ]
        })
    
    return finetuning_data


# Save synthetic dataset (uses shared save_dataset from utils.data_io)
save_dataset(synthetic_dataset, config.output_file)
print(f"Saved {len(synthetic_dataset)} entries to: {config.output_file}")

# Save in fine-tuning format
finetuning_data = convert_to_finetuning_format(synthetic_dataset)
save_dataset(finetuning_data, config.finetuning_file)
print(f"\nSaved {len(finetuning_data)} examples in fine-tuning format")

## Analysis of Generated Data

In [ ]:
import matplotlib.pyplot as plt

# Analyze the synthetic dataset
df = pd.DataFrame(synthetic_dataset)
df['question_length'] = df['question'].str.len()
df['reasoning_length'] = df['reasoning'].str.len()
df['answer_length'] = df['answer'].str.len()

print("\nSYNTHETIC DATASET STATISTICS")
print("="*80)
print(f"Total examples: {len(df)}")
print(f"\nLength statistics:")
print(df[['question_length', 'reasoning_length', 'answer_length']].describe())

# Plot distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df['question_length'].hist(bins=30, ax=axes[0], edgecolor='black', color='blue')
axes[0].set_title('Question Length Distribution')
axes[0].set_xlabel('Characters')
axes[0].set_ylabel('Count')

df['reasoning_length'].hist(bins=30, ax=axes[1], edgecolor='black', color='green')
axes[1].set_title('Reasoning Length Distribution')
axes[1].set_xlabel('Characters')

df['answer_length'].hist(bins=30, ax=axes[2], edgecolor='black', color='orange')
axes[2].set_title('Answer Length Distribution')
axes[2].set_xlabel('Characters')

plt.tight_layout()
plt.savefig('../data/synthetic_dataset_stats.png', dpi=300)
plt.show()

print(f"\nSaved plot to: ../data/synthetic_dataset_stats.png")

## Sample Generated Examples

In [ ]:
# Show some examples
print("\n" + "="*80)
print("SAMPLE SYNTHETIC EXAMPLES")
print("="*80)

for i in range(min(3, len(synthetic_dataset))):
    item = synthetic_dataset[i]
    print(f"\n--- Example {i+1} ---")
    print(f"\nQUESTION:")
    print(item['question'][:300])
    if len(item['question']) > 300:
        print("...")
    
    print(f"\nREASONING:")
    print(item['reasoning'][:300])
    if len(item['reasoning']) > 300:
        print("...")
    
    print(f"\nANSWER:")
    print(item['answer'])
    print("-"*80)

## Combine with Original Dataset (Optional)

In [ ]:
# Option: Combine synthetic with original cleaned data for fine-tuning
combine_with_original = True  # Set to True to combine

if combine_with_original:
    print("\nCombining synthetic with original dataset...")
    
    # Mark original data
    for item in seed_dataset:
        if 'source' not in item or item['source'] == 'synthetic':
            item['source'] = 'original'
    
    # Combine
    combined_dataset = seed_dataset + synthetic_dataset
    
    # Convert to fine-tuning format
    combined_finetuning = convert_to_finetuning_format(combined_dataset)
    
    # Save
    combined_file = '../data/combined_finetuning_format.jsonl'
    save_dataset(combined_finetuning, combined_file)
    
    print(f"\nCombined dataset:")
    print(f"  Original: {len(seed_dataset)}")
    print(f"  Synthetic: {len(synthetic_dataset)}")
    print(f"  Total: {len(combined_dataset)}")
    print(f"  Saved to: {combined_file}")
else:
    print("\nSkipped combining datasets (set combine_with_original=True to enable)")

## Cost Analysis

In [ ]:
# Rough cost estimates (check OpenRouter for exact rates)
COST_PER_1M_TOKENS = {
    'anthropic/claude-sonnet-4': 3.00,
    'openai/gpt-4o': 2.50,
    'qwen/qwen-2.5-32b-instruct': 0.60,
}

cost_per_1m = COST_PER_1M_TOKENS.get(config.generator_model, 1.00)
estimated_cost = (total_tokens / 1_000_000) * cost_per_1m

print("\n" + "="*80)
print("COST ANALYSIS")
print("="*80)
print(f"Model: {config.generator_model}")
print(f"Tokens used: {total_tokens:,}")
print(f"Estimated cost: ${estimated_cost:.2f}")
print(f"Cost per example: ${estimated_cost/len(synthetic_dataset):.4f}")
print("\nNote: Check OpenRouter for exact pricing.")

## Summary

In [ ]:
print("\n" + "="*80)
print("SYNTHETIC DATA GENERATION SUMMARY")
print("="*80)

print(f"\nDATASET:")
print(f"   Seed examples:        {len(seed_dataset)}")
print(f"   Synthetic generated:  {len(synthetic_dataset)}")
print(f"   Total available:      {len(seed_dataset) + len(synthetic_dataset)}")

print(f"\nGENERATION:")
print(f"   Generator model:      {config.generator_model}")
print(f"   Questions per batch:  {config.questions_per_seed}")
print(f"   Seed examples/batch:  {config.num_seed_examples}")

print(f"\nCOST:")
print(f"   Tokens used:          {total_tokens:,}")
print(f"   Estimated cost:       ${estimated_cost:.2f}")

print(f"\nOUTPUT FILES:")
print(f"   {config.output_file}")
print(f"   {config.finetuning_file}")
if combine_with_original:
    print(f"   ../data/combined_finetuning_format.jsonl")
print(f"   ../data/synthetic_dataset_stats.png")

print(f"\nNEXT STEPS:")
print(f"   1. Review sample synthetic examples above")
print(f"   2. Inspect output files for quality")
print(f"   3. Use {config.finetuning_file} or combined file for fine-tuning Qwen 14B")
print(f"   4. Adjust config and regenerate if needed")

print(f"\nWHY THIS WORKS:")
print(f"   - Used {config.generator_model} (more capable than Qwen 14B)")
print(f"   - Generated questions that will challenge Qwen 14B")
print(f"   - Included full reasoning chains for learning")
print(f"   - Diverse examples from different seed combinations")

print("\n" + "="*80)